<a href="https://colab.research.google.com/github/rbvwolf/volt-guardian/blob/main/volt_guardian.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Eğer SHAP Colab'de yüklü değilse, bu satır onu Google sunucusuna anında kurar:
!pip install shap -q

# Alet Çantamızı (Kütüphaneleri) Çağırıyoruz:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import xgboost as xgb
import shap

print("Tüm sistemler aktif! Volt-Guardian XAI projesi kodlamaya hazır.")

Tüm sistemler aktif! Volt-Guardian XAI projesi kodlamaya hazır.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
dosya_yolu = '/content/drive/MyDrive/Volt-Guardian/ev_battery_degradation_v1.csv'
df = pd.read_csv(dosya_yolu)

display(df.head())

print("data set")
df.info()

,Vehicle_ID,Car_Model,Battery_Type,Battery_Capacity_kWh,Vehicle_Age_Months,Total_Charging_Cycles,Avg_Temperature_C,Fast_Charge_Ratio,Avg_Discharge_Rate_C,Driving_Style,Internal_Resistance_Ohm,SoH_Percent,Battery_Status
0,1fb46ae8,Tesla Model 3,NMC,75.0,41,390,21.5,0.51,2.22,Aggressive,0.0362,94.60,Healthy
1,b7ef35aa,Tesla Model 3,NMC,75.0,29,401,18.0,0.62,1.34,Aggressive,0.0333,95.68,Healthy
2,76cb49e0,Ford Mustang Mach-E,NMC,88.0,71,941,18.4,0.78,1.48,Conservative,0.0526,89.80,Healthy
3,456a7aef,Ford Mustang Mach-E,NMC,88.0,57,378,10.8,0.61,0.72,Moderate,0.0314,96.29,Healthy
4,bd758049,Tesla Model 3,NMC,75.0,58,239,30.3,0.89,1.48,Conservative,0.0297,96.75,Healthy


data set
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Vehicle_ID               10000 non-null  object 
 1   Car_Model                10000 non-null  object 
 2   Battery_Type             10000 non-null  object 
 3   Battery_Capacity_kWh     10000 non-null  float64
 4   Vehicle_Age_Months       10000 non-null  int64  
 5   Total_Charging_Cycles    10000 non-null  int64  
 6   Avg_Temperature_C        10000 non-null  float64
 7   Fast_Charge_Ratio        10000 non-null  float64
 8   Avg_Discharge_Rate_C     10000 non-null  float64
 9   Driving_Style            10000 non-null  object 
 10  Internal_Resistance_Ohm  10000 non-null  float64
 11  SoH_Percent              10000 non-null  float64
 12  Battery_Status           10000 non-null  object 
dtypes: float64(6), int64(2), object(5)
memory usage: 1015.8+ KB


In [ ]:
# 1. Çöp kolonları atıyoruz (ID'nin batarya ömrüne etkisi yoktur)
df_temiz = df.drop(columns=['Vehicle_ID'])

# 2. Metin (object) olan değişkenleri 0 ve 1'lerden oluşan sayılara çeviriyoruz
# pd.get_dummies bu işi bizim için otomatik yapar
df_islenmis = pd.get_dummies(df_temiz, columns=['Car_Model', 'Battery_Type', 'Driving_Style'], drop_first=True)

# Yeni verinin röntgenini çekelim (Artık hiç 'object' kalmamış olması lazım)
print("\n--- İşlenmiş Veri Seti Özeti ---")
df_islenmis.info()

# İlk 5 satırın yeni haline bakalım
display(df_islenmis.head())


--- İşlenmiş Veri Seti Özeti ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 16 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Battery_Capacity_kWh           10000 non-null  float64
 1   Vehicle_Age_Months             10000 non-null  int64  
 2   Total_Charging_Cycles          10000 non-null  int64  
 3   Avg_Temperature_C              10000 non-null  float64
 4   Fast_Charge_Ratio              10000 non-null  float64
 5   Avg_Discharge_Rate_C           10000 non-null  float64
 6   Internal_Resistance_Ohm        10000 non-null  float64
 7   SoH_Percent                    10000 non-null  float64
 8   Battery_Status                 10000 non-null  object 
 9   Car_Model_Ford Mustang Mach-E  10000 non-null  bool   
 10  Car_Model_Hyundai Ioniq 5      10000 non-null  bool   
 11  Car_Model_Tesla Model 3        10000 non-null  bool   
 12  Car_Model_Wul

,Battery_Capacity_kWh,Vehicle_Age_Months,Total_Charging_Cycles,Avg_Temperature_C,Fast_Charge_Ratio,Avg_Discharge_Rate_C,Internal_Resistance_Ohm,SoH_Percent,Battery_Status,Car_Model_Ford Mustang Mach-E,Car_Model_Hyundai Ioniq 5,Car_Model_Tesla Model 3,Car_Model_Wuling Air EV,Battery_Type_NMC,Driving_Style_Conservative,Driving_Style_Moderate
0,75.0,41,390,21.5,0.51,2.22,0.0362,94.60,Healthy,False,False,True,False,True,False,False
1,75.0,29,401,18.0,0.62,1.34,0.0333,95.68,Healthy,False,False,True,False,True,False,False
2,88.0,71,941,18.4,0.78,1.48,0.0526,89.80,Healthy,True,False,False,False,True,True,False
3,88.0,57,378,10.8,0.61,0.72,0.0314,96.29,Healthy,True,False,False,False,True,False,True
4,75.0,58,239,30.3,0.89,1.48,0.0297,96.75,Healthy,False,False,True,False,True,True,False


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Hedef (Target - y) Değişkeni: Neyi tahmin edeceğiz?
y = df_islenmis['SoH_Percent']

# 2. Girdiler (Inputs - X): Kopya çektirecek olan 'Battery_Status' ve
# zaten tahmin edeceğimiz 'SoH_Percent' kolonlarını tablodan atıyoruz.
X = df_islenmis.drop(columns=['SoH_Percent', 'Battery_Status'])

# 1. Veriyi Eğitim (%80) ve Test (%20) olarak bölüyoruz
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Yapay zeka modeli eğitiliyor... (Matrisler çarpılıyor)")

# 2. Yapay Zeka Modelini Çağırıyoruz (İçinde 100 adet karar ağacı olan bir orman kuruyoruz)
model = RandomForestRegressor(n_estimators=100, random_state=42)

# 3. EĞİTİMİ BAŞLATIYORUZ (İşte makine burada öğreniyor)
model.fit(X_train, y_train)
print("Eğitim Tamamlandı! 🎉\n")

# 4. TEST AŞAMASI (Modeli, daha önce hiç görmediği 2000 araba ile sınava sokuyoruz)
tahminler = model.predict(X_test)

# 5. KARNE (Hocaya ve Jüriye Sunulacak Metrikler)
hata_payi = mean_absolute_error(y_test, tahminler)
basari_orani = r2_score(y_test, tahminler)

print("--- MODELİN SINAV SONUCU ---")
print(f"Ortalama Hata (MAE)     : % {hata_payi:.2f} (Yani model tahminde bulunurken ortalama sadece bu kadar yanılıyor)")
print(f"Başarı Skoru (R2 Score) : % {basari_orani*100:.2f} (100'e ne kadar yakınsa o kadar kusursuzdur)")

Yapay zeka modeli eğitiliyor... (Matrisler çarpılıyor)
Eğitim Tamamlandı! 🎉

--- MODELİN SINAV SONUCU ---
Ortalama Hata (MAE)     : % 0.29 (Yani model tahminde bulunurken ortalama sadece bu kadar yanılıyor)
Başarı Skoru (R2 Score) : % 98.64 (100'e ne kadar yakınsa o kadar kusursuzdur)


In [ ]:
import pandas as pd

# Kendi hayali aracımızı (veya test aracımızı) yaratıyoruz
# Kolon sıralaması X_train ile BİREBİR aynı olmalıdır.
yeni_arac = pd.DataFrame([{
    'Battery_Capacity_kWh': 75.0,        # Standart Tesla Model 3 bataryası
    'Vehicle_Age_Months': 36,            # 3 yıllık araç
    'Total_Charging_Cycles': 700,        # 500 kere şarj olmuş
    'Avg_Temperature_C': 40.0,           # Hep sıcak havada kullanılmış (Batarya düşmanı)
    'Fast_Charge_Ratio': 0.6,           # Sürücü hep DC hızlı şarj kullanmış (%90)
    'Avg_Discharge_Rate_C': 2.8,         # Agresif kullanım (Gaza çok sert basılmış)
    'Internal_Resistance_Ohm': 0.0480,   # İç direnç yükselmiş (Yıpranma belirtisi)
    'Car_Model_Ford Mustang Mach-E': 1,
    'Car_Model_Hyundai Ioniq 5': 0,
    'Car_Model_Tesla Model 3': 0,        # Aracımız Tesla
    'Car_Model_Wuling Air EV': 0,
    'Battery_Type_NMC': 1,               # NMC tipi batarya
    'Driving_Style_Conservative': 0,     # Sakin sürmüyor
    'Driving_Style_Moderate': 0          # Orta da sürmüyor (Makine bunun Agresif olduğunu anlayacak)
}])

# Modele soruyoruz: "Bu bataryanın yüzde kaç ömrü kalmıştır?"
tahmin_edilen_soh = model.predict(yeni_arac)

print("--- YAPAY ZEKA TAHMİNİ ---")
print(f"Bu aracın batarya sağlığı (SoH) tahminen: % {tahmin_edilen_soh[0]:.2f}")

--- YAPAY ZEKA TAHMİNİ ---
Bu aracın batarya sağlığı (SoH) tahminen: % 90.96


In [16]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# --- 1. AŞAMA: YENİ İNPUTLAR ÜRETİYORUZ (FEATURE ENGINEERING) ---
# Mevcut kolonları matematiksel olarak birleştirip 3 yeni kolon yaratıyoruz
df_islenmis['Thermal_Stress_Index'] = df_islenmis['Avg_Temperature_C'] * df_islenmis['Fast_Charge_Ratio']
df_islenmis['Monthly_Charge_Intensity'] = df_islenmis['Total_Charging_Cycles'] / (df_islenmis['Vehicle_Age_Months'] + 1) # Sıfıra bölünme hatası olmasın diye +1 eklendi
df_islenmis['Power_Stress'] = df_islenmis['Avg_Discharge_Rate_C'] * df_islenmis['Internal_Resistance_Ohm']

print("Yeni kolonlar başarıyla üretildi!")

# --- 2. AŞAMA: X VE Y AYRIMI ---
y = df_islenmis['SoH_Percent']
X = df_islenmis.drop(columns=['SoH_Percent', 'Battery_Status'])

print(f"Yapay zeka artık {X.shape[1]} adet (14 eski + 3 yeni) input ile eğitilecek.\n")

# --- 3. AŞAMA: EĞİTİM VE TEST ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

tahminler = model.predict(X_test)
hata_payi = mean_absolute_error(y_test, tahminler)
basari_orani = r2_score(y_test, tahminler)

print("--- YENİ MODELİN SINAV SONUCU ---")
print(f"Ortalama Hata (MAE)     : % {hata_payi:.2f}")
print(f"Başarı Skoru (R2 Score) : % {basari_orani*100:.2f}")

Yeni kolonlar başarıyla üretildi!
Yapay zeka artık 17 adet (14 eski + 3 yeni) input ile eğitilecek.

--- YENİ MODELİN SINAV SONUCU ---
Ortalama Hata (MAE)     : % 0.29
Başarı Skoru (R2 Score) : % 98.66
